# Exchange Student Expenses Dataset

This dataset is built from my actual expense ledger during my exchange semester in Grenoble, France (August 22, 2025 – January 31, 2026). This notebook preprocesses the raw, multi-currency ledger into a clean, public-ready dataset.

1. **Privacy Filtered**: Removed sensitive columns like specific expense details and payer information. (Indeed, the payer's value is my name)

2. **Date Standardized**: Parsed messy date strings, split date ranges into distinct Start and End dates, and added a Year-Month column for analysis.

3. **Unified Currency (KRW)**: Fetched historical daily exchange rates via yfinance to automatically convert all foreign transactions (EUR, USD, GBP, CHF, TRY) into Korean Won (expense_krw) based on the exact transaction date.

In [11]:
import yfinance as yf
import datetime
import pandas as pd
import re

In [15]:
# 1. Load data
df = pd.read_csv('교환학생 가계부.csv')

# 2. Drop privacy info
df_clean = df.drop(columns=['지출 내용', '결제자'], errors='ignore')

##################################################################
# 3. Handle '지출 날짜' and extract end dates for periods

# Helper function: Extract ONLY the "YYYY년 MM월 DD일" part from any messy string
def extract_clean_date(date_str):
    # Regex: Find exactly 4 digits + 년, 1~2 digits + 월, 1~2 digits + 일
    match = re.search(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', date_str)
    if match:
        # Reformat as "YYYY-MM-DD"
        return f"{match.group(1)}-{match.group(2)}-{match.group(3)}"
    return None

# Handle '지출 날짜' and extract end dates for periods
def parse_start_date(date_str):
    if pd.isna(date_str): return pd.NaT
    date_str = str(date_str)
    
    # Extract the start date (before '→')
    if '→' in date_str:
        date_str = date_str.split('→')[0]
        
    clean_str = extract_clean_date(date_str)
    return pd.to_datetime(clean_str, errors='coerce') if clean_str else pd.NaT

def parse_end_date(date_str):
    if pd.isna(date_str): return pd.NaT
    date_str = str(date_str)
    
    # Extract the end date (after '→') if it exists
    if '→' in date_str:
        date_str = date_str.split('→')[1]
        
        clean_str = extract_clean_date(date_str)
        return pd.to_datetime(clean_str, errors='coerce') if clean_str else pd.NaT
    
    # Return NaT (Null) if it's a single day expense
    return pd.NaT

# Create a new column for the end date
df_clean['지출 종료 날짜'] = df_clean['지출 날짜'].apply(parse_end_date)

# Overwrite the original column with just the start date
df_clean['지출 날짜'] = df_clean['지출 날짜'].apply(parse_start_date)

# Reorder columns: Put '지출 종료 날짜' right next to '지출 날짜'
cols = df_clean.columns.tolist()
cols.insert(1, cols.pop(cols.index('지출 종료 날짜')))
df_clean = df_clean[cols]

# Add '지출 월' column based on the start date (e.g., 2025-07)
df_clean['지출 월'] = df_clean['지출 날짜'].dt.to_period('M')

###############################################################


# 4. Convert '지출 금액': remove currency symbols/commas and convert to float
currency_cols = [col for col in df_clean.columns if '지출 금액' in col]

def clean_currency(val):
    if pd.isna(val): return 0.0  # null == 0 for aggregation
    val_str = str(val)
    # Regex: keep only digits (0-9), minus sign (-), and decimal point (.)
    cleaned_str = re.sub(r'[^\d\.\-]', '', val_str)
    return float(cleaned_str) if cleaned_str else 0.0

for col in currency_cols:
    df_clean[col] = df_clean[col].apply(clean_currency)


# Calculate historical 'expense_krw' based on exact date
print("💱 Fetching historical exchange rates... (This may take a few seconds)")

# 1) Define currencies to fetch (excluding KRW)
# Define currencies to fetch (excluding KRW), and use USD-TRY cross-rate since TRYKRW=X is often unavailable
tickers = {
    'EUR': 'EURKRW=X',
    'USD': 'USDKRW=X',
    'GBP': 'GBPKRW=X',
    'CHF': 'CHFKRW=X',
    'TRY_USD': 'USDTRY=X' # ✅ Fetching TRY per 1 USD
}

# 2) Download historical rates (Fetching recent 2~3 years of data)
# Set progress=False to suppress output messages
rates_df = pd.DataFrame()
for curr, ticker in tickers.items():
    # Load data from 2023 to today
    data = yf.download(ticker, start="2023-01-01", progress=False)['Close']
    if not data.empty:
        rates_df[curr] = data

# Remove timezone from the index (dates) and normalize
rates_df.index = rates_df.index.tz_localize(None).normalize()

# Forward fill missing dates (e.g., weekends/holidays) with the most recent weekday's rate
all_days = pd.date_range(start=rates_df.index.min(), end=datetime.datetime.today())
rates_df = rates_df.reindex(all_days).ffill()

# 3) Function to calculate total KRW for each row
def calculate_expense_krw(row):
    # Base value: keep existing KRW expense
    total_krw = row['지출 금액 KRW'] 
    
    date = row['지출 날짜']
    if pd.isna(date):
        return total_krw

    # Use the most recent rate if the date is in the future
    today = rates_df.index.max()
    target_date = date if date <= today else today

    # Fetch all rates for the specific target date
    try:
        current_rates = rates_df.loc[target_date]
    except KeyError:
        return total_krw

    # A. Calculate for standard currencies (EUR, USD, GBP, CHF)
    for curr in ['EUR', 'USD', 'GBP', 'CHF']:
        col_name = f'지출 금액 {curr}'
        if col_name in row and row[col_name] != 0:
            rate = current_rates[curr]
            # Convert to float to handle potential Series output
            rate = float(rate.iloc[0]) if isinstance(rate, pd.Series) else float(rate)
            total_krw += row[col_name] * rate

    # B. Special handling for TRY using cross-rate (USD/TRY and USD/KRW)
    if '지출 금액 TRY' in row and row['지출 금액 TRY'] != 0:
        try:
            # formula: TRY_amount * (USDKRW_rate / USDTRY_rate)
            usd_krw = float(current_rates['USD'])
            usd_try = float(current_rates['TRY_USD'])
            try_krw_rate = usd_krw / usd_try
            total_krw += row['지출 금액 TRY'] * try_krw_rate
        except:
            pass # Skip if cross-rate calculation fails

    return total_krw

# 4) Apply calculation and create the 'expense_krw' column
df_clean['expense_krw'] = df_clean.apply(calculate_expense_krw, axis=1)

# Drop decimal points and convert to integer
df_clean['expense_krw'] = df_clean['expense_krw'].round().astype(int)

###############################################################

# 5. Save cleaned dataset to CSV
output_filename = 'exchange_expenses_dataset.csv'
df_clean.to_csv(output_filename, index=False, encoding='utf-8-sig')

print("✅ Expected file [exchange_expenses_dataset.csv] has been created with 'expense_krw' column!")

💱 Fetching historical exchange rates... (This may take a few seconds)
✅ Expected file [exchange_expenses_dataset.csv] has been created with 'expense_krw' column!
